# Gate B — K/τ sweep (Kaggle T4×2)

Default K=32, τ=0.5 already lost to the twin (ΔMSE about −2% at step 650). This session sweeps the remaining kill-rule cells: K ∈ {8, 32, 64}, τ ∈ {0.3, 0.5, 0.7}, skipping 32/0.5. Each arm is 15 minutes from the same Taylor init. Never P100. Never bf16.


In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/Caedral-ai/notrehybrid.git"
cwd = Path.cwd()

if (cwd / "setup_kaggle.sh").exists():
    root = cwd
elif (cwd / "notrehybrid" / "setup_kaggle.sh").exists():
    root = cwd / "notrehybrid"
else:
    !git clone --depth 1 {REPO} notrehybrid
    root = cwd / "notrehybrid"

os.chdir(root)
print("repo root:", root)
!bash setup_kaggle.sh

In [ ]:
WORK = "/kaggle/working"
TEACHER = f"{WORK}/teachers/SmolLM2-360M"
TAYLOR = f"{WORK}/checkpoints/gate-b/init-taylor"
CKPT = f"{WORK}/checkpoints/gate-b"
CFG = "configs/smollm2_360m/gate_a.yaml"
print(TEACHER, TAYLOR, CKPT)

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
TEACHER=/kaggle/working/teachers/SmolLM2-360M
TAYLOR=/kaggle/working/checkpoints/gate-b/init-taylor
CKPT=/kaggle/working/checkpoints/gate-b
CFG=configs/smollm2_360m/gate_a.yaml
echo CELL3_START
python -u -m notre.convert.convert_smollm2 --hf HuggingFaceTB/SmolLM2-360M --out "$TEACHER"
python -u -m notre.convert.taylor_calibrate --cfg "$CFG" --output "$TAYLOR" --teacher "$TEACHER" --hf-teacher HuggingFaceTB/SmolLM2-360M
echo TAYLOR_DONE
for slots in 8 32 64; do
  for tau in 0.3 0.5 0.7; do
    if [ "$slots" = 32 ] && [ "$tau" = 0.5 ]; then
      echo "skip slots=32 tau=0.5 (already measured)"
      continue
    fi
    tag="k${slots}-t${tau}"
    echo "SWEEP_START ${tag}"
    python -m notre.convert.transfer \
      --cfg "$CFG" --teacher "$TEACHER" --student-init "$TAYLOR" \
      --ckpt-dir "$CKPT/sweep/${tag}" \
      --cache --slots "$slots" --tau "$tau" \
      --minutes 15 --save-every 100000 --keep-last 1
    echo "SWEEP_DONE ${tag}"
  done
done
echo SWEEP_ALL_DONE


In [ ]:
print("sweep ran in the previous cell")


In [ ]:
from pathlib import Path
root = Path("/kaggle/working/checkpoints/gate-b/sweep")
if not root.exists():
    print("missing", root)
else:
    for path in sorted(root.glob("*/transfer-cache/mse.csv")):
        print("==", path)
        lines = path.read_text().splitlines()
        print("\n".join(lines[:3]))
        print("...")
        print("\n".join(lines[-3:]))
